# Week 2 — Use an LLM as an annotator

**Research task:** Apply the same housing-opinion codebook through OpenRouter and Ollama, then distinguish route agreement from agreement with a human reference.

**Python introduced:** lists, dictionaries, indexing, dictionary keys and equality comparisons.

Work through input → messages → route → call → raw return → parsed output → check. Predict each output before running its cell. The assessed routine below is the same routine printed in the coursebook and task file.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cjbarrie/GenAI_Soc2026/blob/main/workbook/session02/session02_annotation_measurement.ipynb)

Run the setup cell immediately below before doing anything else. Colab supports the OpenRouter call. The Ollama call is completed in local JupyterLab or VS Code because Colab cannot reach the Ollama server on your computer.


## Prepare the notebook environment

Run the next cell before any other code. In Colab it installs the two small Python SDKs and downloads the public course repository. On a local machine it does not install anything silently: it checks that this notebook is using the course environment and gives the exact repair command if it is not.

**What this cell does not do:** Colab cannot run the Ollama server on your laptop. The Ollama call is therefore skipped in Colab and must be completed later in local JupyterLab or on the in-class machine.

For local work, download the complete repository rather than this notebook alone, start it with `uv run jupyter lab`, and follow any `NEXT STEP` printed by the setup cell. The full instructions are in `docs/ENVIRONMENT_SETUP.md` and in the course book's computing chapter.


In [ ]:
SESSION = "session02"

# Run this cell first. It prepares Colab or checks the local Python environment.
import importlib as setup_importlib
import importlib.util as setup_importlib_util
import os as setup_os
import subprocess as setup_subprocess
import sys as setup_sys
from pathlib import Path as SetupPath

try:
    import google.colab as setup_colab  # type: ignore[import-not-found]
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

course_packages = {
    "openrouter": "openrouter>=0.6,<1",
    "ollama": "ollama>=0.6,<1",
}
missing_packages = [
    package_name
    for package_name in course_packages
    if setup_importlib_util.find_spec(package_name) is None
]

if IN_COLAB:
    if missing_packages:
        packages_to_install = [course_packages[name] for name in missing_packages]
        setup_subprocess.run(
            [
                setup_sys.executable,
                "-m",
                "pip",
                "install",
                "--quiet",
                "--disable-pip-version-check",
                *packages_to_install,
            ],
            check=True,
        )
        setup_importlib.invalidate_caches()

    setup_repo = SetupPath(
        setup_os.getenv("COURSE_COLAB_ROOT", "/content/GenAI_Soc2026")
    )
    if not setup_repo.exists():
        setup_subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/cjbarrie/GenAI_Soc2026.git",
                str(setup_repo),
            ],
            check=True,
        )
    COURSE_ROOT = setup_repo
    setup_os.chdir(COURSE_ROOT / "workbook" / SESSION)
else:
    COURSE_ROOT = SetupPath.cwd()
    while not (COURSE_ROOT / "config" / "course_models.json").exists() and COURSE_ROOT != COURSE_ROOT.parent:
        COURSE_ROOT = COURSE_ROOT.parent
    if missing_packages:
        missing_text = ", ".join(missing_packages)
        raise ModuleNotFoundError(
            f"This notebook is using a Python environment without: {missing_text}.\n\n"
            "Close Jupyter. Open a terminal in the GenAI_Soc2026 repository and run:\n"
            "    uv sync\n"
            "    uv run jupyter lab\n\n"
            "In VS Code, select the Python interpreter inside the repository's .venv folder."
        )
    if not (COURSE_ROOT / "config" / "course_models.json").exists():
        raise FileNotFoundError(
            "The complete GenAI_Soc2026 repository could not be found. A notebook "
            "downloaded by itself is not enough for local work. Download or clone the "
            "repository, open a terminal in that folder, and run: uv run jupyter lab"
        )

course_root_text = str(COURSE_ROOT)
if course_root_text not in setup_sys.path:
    setup_sys.path.insert(0, course_root_text)

still_missing = [
    package_name
    for package_name in course_packages
    if setup_importlib_util.find_spec(package_name) is None
]
if still_missing:
    raise ModuleNotFoundError(
        "Setup did not make these packages available: " + ", ".join(still_missing)
    )

print("Environment:", "Google Colab" if IN_COLAB else "local course environment")
print("Python executable:", setup_sys.executable)
print("Course root:", COURSE_ROOT)
print("Working folder:", SetupPath.cwd())
print("OpenRouter SDK: ready")
print("Ollama Python SDK: ready")
if IN_COLAB:
    print("Ollama model call: skipped here; run it from local JupyterLab")
else:
    import json as setup_json
    import ollama as setup_ollama

    setup_config = setup_json.loads(
        (COURSE_ROOT / "config" / "course_models.json").read_text()
    )
    setup_local_model = setup_config["local"]["model"]
    try:
        setup_models = setup_ollama.list().models
        setup_model_names = [
            getattr(item, "model", None) or getattr(item, "name", None)
            for item in setup_models
        ]
        print("Ollama server: reachable at localhost:11434")
        if setup_local_model in setup_model_names:
            print("Course local model: ready —", setup_local_model)
        else:
            print("Course local model: NOT INSTALLED —", setup_local_model)
            print("NEXT STEP: open a terminal and run: ollama pull " + setup_local_model)
    except Exception as setup_error:
        print("Ollama server: NOT REACHABLE")
        print("NEXT STEP: start the Ollama application, then run: ollama list")
        print("Diagnostic:", str(setup_error).splitlines()[0])


In [ ]:
import json
import os
from getpass import getpass
from pathlib import Path

import ollama
from openrouter import OpenRouter

ROOT = Path.cwd()
while not (ROOT / "config" / "course_models.json").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

config = json.loads((ROOT / "config" / "course_models.json").read_text())
HOSTED_MODEL = config["hosted"]["model"]
LOCAL_MODEL = config["local"]["model"]

if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter course key (hidden): ")

print("Hosted model:", HOSTED_MODEL)
print("Local model:", LOCAL_MODEL)

## Store one comment, the codebook and a human reference label

`comment` is the text to classify. `codebook` is a dictionary mapping each permitted label to its definition. `human_label` is a separate reference judgment; it is not sent to the model. `codebook["UNCLEAR"]` uses a key to retrieve one definition and demonstrates how dictionaries differ from lists.


In [ ]:
comment = "I support the plan if rents remain affordable."
codebook = {
    "SUPPORT": "unconditional support for the proposal",
    "OPPOSE": "unconditional opposition to the proposal",
    "UNCLEAR": "conditional, mixed, procedural, or insufficient evidence",
}
human_label = "UNCLEAR"
print(comment)
print(codebook["UNCLEAR"])

## Construct the exact message list used by both routes

`json.dumps(codebook)` turns the Python dictionary into readable text that can be placed inside the prompt. The `+` signs join codebook, instruction and comment. The resulting `messages` list contains one user message. `[0]` retrieves the first list item; `["content"]` then retrieves that dictionary's content field.


In [ ]:
prompt = (
    "Apply this codebook: " + json.dumps(codebook) +
    "\nReturn only SUPPORT, OPPOSE, or UNCLEAR.\nComment: " + comment
)
messages = [{"role": "user", "content": prompt}]
print(messages[0]["content"])

## Make and unpack the OpenRouter call

The inputs are the hosted model, the shared message list and zero temperature. `[0]` chooses the first returned choice, `.message.content` retrieves its text, `.strip()` removes spare whitespace and `.upper()` standardizes letter case. The final output, `hosted_label`, remains a string and may still fall outside the codebook.


In [ ]:
with OpenRouter(api_key=os.environ["OPENROUTER_API_KEY"]) as client:
    hosted_response = client.chat.send(
        model=HOSTED_MODEL, messages=messages, temperature=0,
    )
hosted_choice = hosted_response.choices[0]
hosted_raw = hosted_choice.message.content
print("OpenRouter raw return:", hosted_raw)
hosted_label = hosted_raw.strip().upper()
print("OpenRouter label:", hosted_label)

## Make and unpack the Ollama call

The same messages enter Ollama. Its temperature sits inside an `options` dictionary and its returned text is found at `local_response.message.content`. Standardizing case makes the two strings easier to compare; it does not repair a substantively wrong label.


In [ ]:
if IN_COLAB:
    local_response = None
    local_raw = None
    local_label = None
    print("Ollama was not called: Colab cannot reach the local Ollama server.")
else:
    local_response = ollama.chat(think=False, 
        model=LOCAL_MODEL,
        messages=messages,
        options={"temperature": 0},
    )
    local_raw = local_response.message.content
    print("Ollama raw return:", local_raw)
    local_label = local_raw.strip().upper()
    print("Ollama label:", local_label)


## Compare route agreement and human-reference agreement

Each `==` asks whether two strings are equal and returns `True` or `False`. `routes_agree` compares the two models. The other two values compare each model with the independently supplied human reference. These are three distinct questions, so they are stored and printed separately.


In [ ]:
routes_agree = None if local_label is None else hosted_label == local_label
hosted_matches_human = hosted_label == human_label
local_matches_human = None if local_label is None else local_label == human_label
print("Routes agree:", routes_agree)
print("OpenRouter matches human label:", hosted_matches_human)
print("Ollama matches human label:", local_matches_human)

# ONE CHANGE: replace comment with
# "I oppose the rezoning proposal because it will displace tenants."
# Then rerun from the message-construction cell.


## Methodological check

Agreement between the routes is a reproducibility observation. Agreement with one human label is criterion evidence. Neither alone proves that the codebook measures the construct well. If an Ollama comparison prints `None`, that means the local route was deliberately not run in Colab; it is not disagreement between two models.

## Completion recording

Run both routes on the changed opposition comment. Explain the comment string, codebook dictionary, message list, `[0]`, each response path and all three comparisons. State whether disagreement is between routes, with the human reference, or both.

Explain every input and output aloud. Never show the shared key. If you began in Colab, complete the Ollama portion locally or on the in-class machine.
